In [1]:
#%pip install mlflow==2.7.0 scikit-learn pandas
import mlflow, json, os, pandas as pd
from sklearn.metrics import classification_report, accuracy_score


In [2]:
ROOT = os.getcwd()
RESP_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\responses.csv")            # your responses file
HUMAN_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\human_rubric_template.csv")      # human rubric file produced earlier
EVAL_CSV = os.path.join(ROOT, "G:\\Resume-Matcher\\experiments\\prompts\\results\\evaluation_results.csv")           # quantitative metrics CSV you created
REPORT_MD = os.path.join(ROOT, "experiments/prompts/prompt_report.md")


In [5]:
# load responses (long format with columns: id, strategy, assigned_label, ground_truth)
df = pd.read_csv(RESP_CSV)
# convert wide -> long
rows = []
for _, r in df.iterrows():
    rows.append({"id": r["id"], "strategy":"zero", "response_text": r["zero_text"], "assigned_label": r["zero_label"], "ground_truth": r["ground_truth"]})
    rows.append({"id": r["id"], "strategy":"few",  "response_text": r["few_text"],  "assigned_label": r["few_label"],  "ground_truth": r["ground_truth"]})
    rows.append({"id": r["id"], "strategy":"adv",  "response_text": r["adv_text"],  "assigned_label": r["adv_label"],  "ground_truth": r["ground_truth"]})
long = pd.DataFrame(rows)

# normalize labels to High/Medium/Low
def norm(x):
    if pd.isna(x): return x
    s = str(x).lower()
    if "high" in s: return "High"
    if "medium" in s: return "Medium"
    if "low" in s: return "Low"
    return x
long["assigned_label"] = long["assigned_label"].apply(norm)
long["ground_truth"] = long["ground_truth"].apply(norm)

# compute per-strategy accuracy + classification report
summary = {}
for strat, g in long.groupby("strategy"):
    acc = (g["assigned_label"] == g["ground_truth"]).mean()
    report = classification_report(g["ground_truth"], g["assigned_label"], zero_division=0, output_dict=True)
    summary[strat] = {"accuracy": float(acc), "counts": g["assigned_label"].value_counts().to_dict(), "report": report}

# save summary
OUT_SUM = "G:/Resume-Matcher/experiments/prompts/quant_summary.json"

os.makedirs(os.path.dirname(OUT_SUM) or ".", exist_ok=True)
with open(OUT_SUM, "w") as f:
    json.dump(summary, f, indent=2)

# optional: write the long table so human rubric is ready
long.to_csv("human_rubric_filled_from_responses.csv", index=False)

print("Done — summary saved to", OUT_SUM)


Done — summary saved to G:/Resume-Matcher/experiments/prompts/quant_summary.json


In [ ]:
mlflow.set_experiment("prompt_eval")
with mlflow.start_run(run_name="prompt_strategies_local"):
    # log per-strategy metrics
    for strat, stats in summary.items():
        mlflow.log_metric(f"{strat}_accuracy", stats["accuracy"])
        # log counts as metrics
        for label,count in stats["counts"].items():
            mlflow.log_metric(f"{strat}_count_{label}", int(count))
    # log artifacts (responses, human rubric, quant summary)
    mlflow.log_artifact(RESP_CSV, artifact_path="responses")
    mlflow.log_artifact(HUMAN_CSV, artifact_path="human_rubric")
    mlflow.log_artifact(os.path.join("G:\\Resume-Matcher\\experiments\\prompts\\quant_summary.json"), artifact_path="summary")
    # optional: save classification reports per strategy
    for strat, stats in summary.items():
        rep_path = f"G:/Resume-Matcher/experiments/prompts/{strat}_classif.json"
        with open(rep_path,"w") as f: json.dump(stats["classification_report"], f, indent=2)
        mlflow.log_artifact(rep_path, artifact_path=f"classif_reports/{strat}")
print("MLflow logging done. Run `mlflow ui` and open http://localhost:5000 to view.")


FileNotFoundError: [Errno 2] No such file or directory: 'g:\\Resume-Matcher\\experiments\\prompts\\experiments/prompts/adv_classif.json'